# UK Solar Siting — XGBoost Pipeline
Run the full pipeline on Google Colab.

**Prerequisites:** Upload `uk-solar-siting.zip` to your Google Drive root.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Unzip project
!cp /content/drive/MyDrive/uk-solar-siting.zip /content/
!cd /content && unzip -qo uk-solar-siting.zip
%cd /content/uk-solar-siting

In [ ]:
# 3. Install dependencies
!pip install -q geopandas xgboost rasterio xarray netCDF4 folium tqdm pyyaml scipy

In [ ]:
# 3.5 Auto-detect GPU and update config
import subprocess, yaml

gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print(f"GPU available: {gpu}")

if gpu:
    cfg_path = 'configs/config.yaml'
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg['model']['xgb_params']['tree_method'] = 'gpu_hist'
    cfg['model']['xgb_params']['device'] = 'cuda'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("Config updated: tree_method=gpu_hist, device=cuda")

In [ ]:
# 4. Verify data files
from pathlib import Path
raw = Path('data/raw')
expected = ['repd.csv', 'uk_boundary.shp', 'protected_areas.shp',
            'osm_substations.shp', 'osm_roads.shp', 'agricultural_land.shp',
            'os_terrain50.tif', 'era5_uk_solar.nc']
for f in expected:
    p = raw / f
    status = f'{p.stat().st_size / 1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'  {f:30s} {status}')

## Phase 1 — Generate Labelled Data

In [ ]:
from src.utils import load_config
from src.phase1_labeling.combine_samples import combine_samples

cfg = load_config()
labeled = combine_samples(cfg)
labeled.head(10)

## Phase 2 — Feature Engineering

In [ ]:
from src.phase2_features.build_feature_matrix import build_feature_matrix

matrix = build_feature_matrix(cfg)
matrix.describe()

## Phase 3 — Model Training & Evaluation

In [ ]:
from src.phase3_model.train import train_model
from src.phase3_model.evaluate import evaluate_model

model = train_model(cfg)
metrics = evaluate_model(cfg)

In [ ]:
# Display charts
from IPython.display import Image, display
from src.utils import resolve_path

display(Image(filename=str(resolve_path(cfg['paths']['roc_curve_png']))))
display(Image(filename=str(resolve_path(cfg['paths']['feature_importance_png']))))

## Phase 4 — Grid Prediction & Heatmap

In [ ]:
from src.phase4_prediction.predict import predict_grid

predictions = predict_grid(cfg)
predictions.describe()

In [ ]:
from src.phase4_prediction.heatmap import generate_heatmap

m = generate_heatmap(cfg)
m  # display interactive map in notebook

In [ ]:
# Copy results back to Drive
!cp -r data/output /content/drive/MyDrive/uk-solar-siting-output/
print('Results saved to Google Drive: uk-solar-siting-output/')